# SmartBite portion model: grams from one overhead photo
Trains `smartbite_cv.portion` (EfficientNet-B0, log-mass + uncertainty) on Nutrition5k overhead RGB (official depth split), after checking the Kaggle mirror against Google's bucket.

In [ ]:
# SmartBite food recognition - training run (Kaggle GPU or Colab)
import os, sys, glob, json, shutil, subprocess, zipfile, time
CODE_SLUG, DATA_CFG, CLASSES_CFG, TRAIN_EXTRA = "smartbite-cv-code-v2", "configs/data_v2.yaml", "configs/classes_v2.yaml", ""
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working/SmartBite" if ON_KAGGLE else "/content/SmartBite"
OUT = "/kaggle/working/outputs" if ON_KAGGLE else "/content/outputs"
CACHE = "/tmp/smartbite_cache"

def run(cmd):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=WORK)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

if ON_KAGGLE and not os.path.exists(f"{WORK}/smartbite_cv"):
    # The private code dataset is either auto-extracted or still a zip; handle both. Look in its own
    # mount first: a recursive search over every attached image dataset takes minutes.
    # Mounts are /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>.
    mounts = glob.glob(f"/kaggle/input/{CODE_SLUG}") + glob.glob(f"/kaggle/input/*/*/{CODE_SLUG}")
    inits = [i for m in mounts for i in glob.glob(f"{m}/**/smartbite_cv/__init__.py", recursive=True)]
    if inits:
        shutil.copytree(os.path.dirname(os.path.dirname(inits[0])), WORK, dirs_exist_ok=True)
    else:
        zips = [z for m in mounts for z in glob.glob(f"{m}/**/smartbite_code.zip", recursive=True)]
        assert zips, "attach the smartbite-cv-code dataset"
        zipfile.ZipFile(zips[0]).extractall(WORK)
# Colab: upload/clone the repo to /content/SmartBite and put the raw datasets under
# /content/SmartBite/data/raw/<same folder names as configs/data.yaml>, then set PROFILE = "local".
PROFILE = "kaggle" if ON_KAGGLE else "local"
os.makedirs(OUT, exist_ok=True)
# No internet needed: timm comes from the bundled wheel, ImageNet weights from offline/*.safetensors.
whl = glob.glob(f"{WORK}/offline/timm-*.whl")
if ON_KAGGLE and not whl:
    raise RuntimeError("offline/ missing: the attached smartbite-cv-code dataset version is stale; re-run")
run(f"pip install -q --no-index --no-deps {whl[0]}" if whl else "pip install -q timm onnx onnxruntime")
run("nvidia-smi || true")
run("python -c \"import torch, timm; print('torch', torch.__version__, 'timm', timm.__version__)\"")
import torch
assert torch.cuda.is_available(), "No GPU attached: enable the GPU accelerator (Kaggle needs a phone-verified account)"
try:
    import onnxruntime  # noqa: F401
    HAVE_ORT = True
except ImportError:
    HAVE_ORT = False  # ONNX export + latency are then done locally from best.pt
print("onnxruntime available:", HAVE_ORT)


In [ ]:
# Portion (grams) model on Nutrition5k overhead photos.
# 1. The images come from a Kaggle mirror: check a random sample against Google's official bucket (MD5).
import base64, hashlib, random, urllib.parse, urllib.request
# depth-limited: a recursive search would walk every file of the 23 GB mirror
marker = [m for depth in range(4) for m in glob.glob("/kaggle/input/" + "*/" * depth + "imagery/realsense_overhead")]
assert marker, "attach a Nutrition5k mirror with imagery/realsense_overhead"
N5K = os.path.dirname(os.path.dirname(marker[0]))
test_ids = [l.strip() for l in open(f"{WORK}/data/raw/nutrition5k/dish_ids/splits/depth_test_ids.txt") if l.strip()]
have = [d for d in test_ids if os.path.exists(f"{N5K}/imagery/realsense_overhead/{d}/rgb.png")]
sample = random.Random(20261031).sample(have, min(100, len(have)))
bad, checked = [], 0
for d in sample:
    name = urllib.parse.quote(f"nutrition5k_dataset/imagery/realsense_overhead/{d}/rgb.png", safe="")
    try:
        with urllib.request.urlopen(f"https://storage.googleapis.com/storage/v1/b/nutrition5k_dataset/o/{name}?fields=md5Hash", timeout=30) as r:
            official = json.load(r)["md5Hash"]
    except Exception as exc:
        print("could not reach the official bucket:", exc)
        break
    local = base64.b64encode(hashlib.md5(open(f"{N5K}/imagery/realsense_overhead/{d}/rgb.png", "rb").read()).digest()).decode()
    checked += 1
    if local != official:
        bad.append(d)
print(f"mirror check: {checked} sampled test images compared with the official bucket, {len(bad)} differ; "
      f"{len(have)}/{len(test_ids)} test dishes have an overhead photo in the mirror")
assert not bad, f"mirror images differ from the official dataset: {bad[:5]}"
os.makedirs(f"{OUT}/portion", exist_ok=True)
json.dump({"checked": checked, "mismatches": bad, "test_dishes_with_photo": len(have), "test_dishes": len(test_ids)},
          open(f"{OUT}/portion/mirror_check.json", "w"), indent=2)
# 2. Train and evaluate (test split used once).
weights = ("--set model.pretrained_file=offline/efficientnet_b0.ra_in1k.safetensors"
           if os.path.exists(f"{WORK}/offline/efficientnet_b0.ra_in1k.safetensors") else "")
t0 = time.time()
run(f"python -m smartbite_cv.portion train --config configs/train/portion.yaml --profile kaggle {weights} --out runs/portion")
run("python -m smartbite_cv.portion evaluate --run runs/portion --profile kaggle")
print(f"portion model: {(time.time() - t0) / 60:.1f} min")
for f in ["best.pt", "history.csv", "config_resolved.json"]:
    shutil.copy(f"{WORK}/runs/portion/{f}", f"{OUT}/portion/")
shutil.copytree(f"{WORK}/runs/portion/eval", f"{OUT}/portion/eval", dirs_exist_ok=True)
if ON_KAGGLE:
    shutil.rmtree(WORK, ignore_errors=True)
print(open(f"{OUT}/portion/eval/portion_metrics.json").read())
